# Data Structures and Plotting

Initially, we require several libraries and scientific modules for our tasks. Run the cell.

In [ ]:
# Standard libraries
import os


# Scientific libraries
import numpy as np
import xarray as xr

#Visualization libraries

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

import emoji




E.g. xarray is a Python library designed for loading, analyzing, and managing scientific data efficiently. It's especially useful for working with multidimensional data, common in fields like meteorology, oceanography, and climate science.

Functions and Methods:

    With xarray, we can perform statistical calculations such as mean (mean), standard deviation (std), and many others directly on the data. These methods simplify complex data operations.

Handling Multiple Datasets:

    xarray also supports loading and analyzing multiple datasets at once.

Dot Notation:

    Methods in xarray are accessed using dot notation, making code more readable and organized. For example, to calculate the mean of a dataset, we simply write dataset.mean().

In [ ]:
# This is a command specific to Jupyter Notebooks that ensures Matplotlib plots are embedded
# and displayed directly within the notebook interface, independent of the Jupyter version.
%matplotlib inline

In [ ]:
print(emoji.emojize('Python is :thumbs_up:'))

### Working with netCDF data 

**Downloading the 2023 Monthly Wind Data**

If not done yet, navigate to the Copernicus Marine Service website to access the 2023 monthly wind data. Specifically, download the 12 NetCDF files for 2024 from the following link:
https://data.marine.copernicus.eu/product/WIND_GLO_PHY_CLIMATE_L4_MY_012_003/files?subdataset=cmems_obs-wind_glo_phy_my_l4_P1M_202411&path=WIND_GLO_PHY_CLIMATE_L4_MY_012_003%2Fcmems_obs-wind_glo_phy_my_l4_P1M_202411%2F2024%2F

Once downloaded, store the data in `OceanographicDataProcessingCourse/Data/Wind`

In the cell below, please provide the file path to where you've stored the downloaded data. This will allow for the data to be accessed and processed in the subsequent steps. You can use a relative or absolute path. For example: `C:/PATH/TO/FILE` on any operating system. Today, we start with the data of January 2023.



In [ ]:
## datapath and filename
datapath = '../Data/Wind'
filename = "cmems_obs-wind_glo_phy_my_l4_P1M_202401.nc"


Since geographic data files can often be very large, when we first open our data file in xarray it simply loads the metadata associated with the file. We can then view summary information about the contents of the file before deciding whether we’d like to load some or all of the data into memory ( xarray allows for a quick view of the dataset's metadata without loading the full data, but once the data is accessed, it will be loaded.). Run the next cell

In [ ]:
#run the cell
full_path = os.path.join(datapath, filename)
ds = xr.open_dataset(full_path)

An xarray has typically the following components:  
data : data array ( values)  
coords : dictonary which shows dimensions with corresponding coordinates and data types
attrs : dictionary with metadata and attributes  
Have a look into the xarray ds by running the next cell.

In [ ]:
## run the cell 
ds

Alternatively you can use the `print()`-Statement, which is also helpful when you work with excecutable Python Scripts

In [ ]:
#run the cell
print(ds)


You've already observed the variables within the dataset. Another method to display the variables included in the dataset is to simply use `list()` on `data_vars`.

In [ ]:
#run the cell
list(ds.data_vars)

From this point, you'll have a clear overview of what's contained in the data. You'll also be able to see how the data is distributed both temporally and spatially. If you can't immediately discern the resolution of the data, the following code snippet will assist you:

In [ ]:
# run the cell
if 'lon' in ds.dims and len(ds.lon) > 1:
    print("Longitude resolution:", ds.lon.values[1] - ds.lon.values[0])
if 'lat' in ds.dims and len(ds.lat) > 1:
    print("Latitude resolution:", ds.lat.values[1] - ds.lat.values[0])
if 'time' in ds.dims and len(ds.time) > 1:
    print("Temporal resolution:", ds.time.values[1] - ds.time.values[0])

In [ ]:
# or 
ds.lon.diff(dim = 'lon')#.plot()

You might have noticed the absence of an output for temporal resolution. Why might that be? Although the dataset has a `'time'` dimension, it doesn't truly have a temporal resolution since there's only one time value. This indicates that we have only one timestep.

You can see, which variables the data set includes and which dimension they have. We are interested in the u and v variable contained within that xarray dataset and named here `eastward_wind` and `northward_wind`:

In the xarray library, a dataset (often denoted as ds) represents an in-memory on-disk database of arrays. These arrays can be thought of as variables in the dataset. There are two primary ways to access these variables:

1. Attribute-style access: `ds.variable_name`
2. Dictionary-style access: `ds['variable_name']`

Attribute access is shorter but might not always work, especially with invalid Python names (e.g. '123' or 'print'). Dictionary access is more universal and works with any variable name.

Testing both access' by running the next two cells.


In [ ]:
# run the cell
ds.eastward_wind

In [ ]:
#run the cell
ds['eastward_wind']

Both lines should produce the same output, assuming that `eastward_wind` is a valid variable in your dataset. If not, you'd get an error.

In summary, both methods are valid ways to access xarray Dataset variables, and which one to use often comes down to personal preference, the specific situation, and the variable names you're working with.

## Visualization

With `xarray`'s built-in plotting functionality, we can easily visualize DataArrays. Here, we are plotting `ds.eastward_wind` at the time index `time = 0` for all values of latitude and longitude using `ds.eastward_wind[0,:,:]`. If this dataset contained data for multiple months, we would specify the desired time index accordingly.

In [ ]:
# run the cell
# Plot eastward wind using xarray's built-in plotting functionality
ds.eastward_wind[0,:,:].plot(cmap='coolwarm', figsize=(12, 5)) #plasma, viridis, coolwarm, jet


This plot already looks quite impressive! We can observe the zonal wind velocity, with positive amplitudes in the mid-latitudes and negative amplitudes in the higher latitudes, as well as between -20° and 20°. However, now we also want to visualize the continents and add the spatial grid to the plot.


In [ ]:
#run the cell

# Mask for valid data
mask = ds.number_of_observations.notnull()

# Create the figure and axis with geographic projection
fig, ax = plt.subplots(
    figsize=(12, 5),
    subplot_kw={'projection': ccrs.PlateCarree()}  # geographic coordinates (lon/lat)
)

# Plot eastward wind field from the xarray dataset
ds.eastward_wind[0, :, :].where(mask).plot(
    ax=ax,
    cmap='coolwarm',
    transform=ccrs.PlateCarree()  # data is already in lon/lat
)

# Add coastlines and borders
ax.coastlines(linewidth=1, color='black')        # color of coastlines
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')  # color of borders
ax.add_feature(cfeature.LAND, facecolor='white')  # color of landmasses

# Add gridlines with labels
gl = ax.gridlines(
    draw_labels=True,
    linestyle='--',
    color='gray',
    alpha=0.7
)
gl.top_labels = False
gl.right_labels = False

plt.show()


First, we create the figure and axis with a geographic projection. We plot the eastward wind using xarray's built-in plotting functionality, masking out invalid data.
Then we add landmasses and borders to give the map geographical context.
Finally, we add gridlines and style them for better readability.

When you take a closer look at the code snippet above, where do you think you could change the color of the landmasses and the borders, for example?

**1. Exercise: Copy the code from above and modify it to change the color of the landmasses and borders. Experiment and see how different color schemes affect the visualization.**



In [ ]:
#copy the code from above change the relevant paramters



The xarray.plot() function expects regularly gridded data on a flat, linear axis, which could result in distorted representations of the Earth. In flat projections, the sizes and shapes of geographical features are not accurate — for example, landmasses near the poles appear much larger than they actually are. To create more realistic representations of the Earth, we could use map projections, such as the Robinson projection.


In [ ]:
#run the cell

#  Create the figure and axis with Robinson projection
fig, ax = plt.subplots(
    figsize=(12, 5),
    subplot_kw={'projection': ccrs.Robinson()}  # <- just change this line
)

# Plot eastward wind field from the xarray dataset
ds.eastward_wind[0, :, :].where(mask).plot(
    ax=ax,
    cmap='coolwarm',
    levels=np.linspace(-10, 10, 21),  # <- adjust levels for eastward wind        
    transform=ccrs.PlateCarree()  # data still in lon/lat coordinates
)

# Add coastlines and borders
ax.coastlines(linewidth=1, color='black')        # color of coastlines
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')  # color of borders
ax.add_feature(cfeature.LAND, facecolor='white')  # color of landmasses

# Add gridlines with labels
gl = ax.gridlines(
    draw_labels=True,
    linestyle='--',
    color='gray',
    alpha=0.7
)
gl.top_labels = False
gl.right_labels = False

plt.show()



Cartopy converts geographic coordinates (latitude and longitude) into a chosen map projection, such as the Robinson projection, allowing for accurate visual representation of the Earth’s curved surface. In contrast, the xarray.plot() function operates on 2D Cartesian grids and does not inherently apply geographic map projections.

**2. Exercise: Copy the code from above and**:

    1. Plot the northward_wind component.
    2. Change the color of the landmasses.
    3. Modify the color, linewidth, and spacing of the parallels and meridians (gridlines). 
    4. Consider what else to adapt for a new variable (e.g., color scale, units, or labels).

For example, you can use  `gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 40))`   and `gl.ylocator = mticker.FixedLocator(np.arange(-90,   91, 30))` to modify the spacing of the gridlines

In [ ]:
# copy the code from above and modify



Now that you've explored various plotting techniques, you have a basic understanding of how zonal (eastward) and meridional (northward) wind velocities look. Wind, as a vector, has both a northward and eastward component, which are typically combined and represented as wind vectors. You might recognize this from weather apps, where wind direction and strength are often shown using arrows.

We have plotted the wind components separately, but typically, wind data is represented using vector arrows to visually display both speed and direction. To do this, we can use the quiver() function:

In [ ]:
# run this cell

fig, ax = plt.subplots(
    figsize=(12, 7),
    subplot_kw={'projection': ccrs.PlateCarree()}  # matches lon/lat grid
)

# Map context: coastlines, land, borders 
ax.add_feature(cfeature.LAND, facecolor='white')          # land color
ax.coastlines(resolution='110m', linewidth=0.8)           # coastlines
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')

# Gridlines with labels (parallels/meridians) 
gl = ax.gridlines(draw_labels=False, linestyle='--', color='gray', linewidth=0.5, alpha=0.8)
gl.top_labels = False
gl.right_labels = False
ax.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(-90,   91, 30), crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
ax.yaxis.set_major_formatter(LatitudeFormatter())

# Prepare lon/lat and wind components for quiver 
lon = ds.coords['lon'].values
lat = ds.coords['lat'].values
lon2d, lat2d = np.meshgrid(lon, lat)

# Select one time slice and a sample step in horizontal direction
sample_step = 1  # adjust step 
U = ds.eastward_wind.isel(time=0).values[::sample_step, ::sample_step]
V = ds.northward_wind.isel(time=0).values[::sample_step, ::sample_step]
X = lon2d[::sample_step, ::sample_step]
Y = lat2d[::sample_step, ::sample_step]

# Quiver on a map: provide data CRS via 'transform' 
q = ax.quiver(
    X, Y, U, V,
    transform=ccrs.PlateCarree(),  # data are in lon/lat
    scale=500,                     # adjust arrow length scaling
    width=0.0025                   # arrow line width
)


# Title 
time_str = str(ds.coords['time'].values[0])[:10]
ax.set_title(f'Wind Vector Plot (Eastward & Northward) at {time_str} ({ds.eastward_wind.units})')

plt.show()


Our black map is caused by overplotting: with 0.25° data we try to render ~1 Million (720*1440 = 1036800) arrows, which saturates the figure. 

# Data Reduction Techniques - Exploring Coarsen and Slice 

The fix is to reduce vectors:
    - Slicing: pick every nth point (fast, simple).
    - Downsampling (coarsen): average onto a coarser grid (smoother, preserves patterns).


Let's start by testing slicing.
In the next example, we reuse the previous code and vary the sample_step parameter to observe how it affects the plot.

In [ ]:
# run this cell

fig, ax = plt.subplots(
    figsize=(12, 7),
    subplot_kw={'projection': ccrs.PlateCarree()}  # matches lon/lat grid
)

# Map context: coastlines, land, borders 
ax.add_feature(cfeature.LAND, facecolor='white')          # land color
ax.coastlines(resolution='110m', linewidth=0.8)           # coastlines
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')

# Gridlines with labels (parallels/meridians) 
gl = ax.gridlines(draw_labels=False, linestyle='--', color='gray', linewidth=0.5, alpha=0.8)
gl.top_labels = False
gl.right_labels = False
ax.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(-90,   91, 30), crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
ax.yaxis.set_major_formatter(LatitudeFormatter())

# Prepare lon/lat and wind components for quiver 
lon = ds.coords['lon'].values
lat = ds.coords['lat'].values
lon2d, lat2d = np.meshgrid(lon, lat)
###################################################################################
# Select one time slice and a sample step in horizontal direction

sample_step = 20  # adjust step 

U = ds.eastward_wind.isel(time=0).values[::sample_step, ::sample_step]
V = ds.northward_wind.isel(time=0).values[::sample_step, ::sample_step]
X = lon2d[::sample_step, ::sample_step]
Y = lat2d[::sample_step, ::sample_step]
####################################################################################
# Quiver on a map: provide data CRS via 'transform' 
q = ax.quiver(
    X, Y, U, V,
    transform=ccrs.PlateCarree(),  # data are in lon/lat
    scale=500,                     # adjust arrow length scaling
    width=0.0025                   # arrow line width
)


# Title 
time_str = str(ds.coords['time'].values[0])[:10]
ax.set_title(f'Wind Vector Plot (Eastward & Northward) at {time_str} ({ds.eastward_wind.units})')

plt.show()

The direction of the arrow represents the wind direction, while the length of the arrow indicates the wind speed.

**3. Exercise: Copy the code from above and experiment with slicing. Try selecting every 10th or 50th point, and vary the slicing for the zonal latitude and longitude direction. What else do you need to consider? (For example, when slicing differently for each direction, remember to adjust latitude and longitude accordingly.)**

You can also use `ds.eastward_wind[0, slice(None, None, Step), slice(None, None, Step)]`, where Step can be set to 20, 50, 10, or any other value depending on how much you want to slice the data.

In [ ]:
# your code here 



Great! To create a more intelligible visualization, you employed data slicing. It's important to remember that slicing can omit certain values, possibly leading to a loss of information. An alternative is to use spatial averaging, which can be seamlessly achieved using the `coarsen method`. 


First, we need to calculate the absolute wind speed: 
To find the wind speed, `U`, from the eastward (u) and northward (v) components, use:

$U = \sqrt{u^2 + v^2}$


**4. Exercise: Use  `np.sqrt()`in numpy to compute `U`. Plug in your data for u and v. Ensure the units and name attributes in your xarray data array are correct. Update the metadata attributes of your xarray data array (`wind_speed.attrs['units'] = 'UNIT' , wind_speed.attrs['long_name'] = 'NAME OF VARIABLE'`). Then, plot wind speed with your preferred method.**


Tip: To square a value in Python, use `**`

In [ ]:
## your computation and plot of wind speed here


#### Common Statistical Operations in Python

Libraries like xarray, numpy, and pandas allow statistical operations such as .mean(), .std(), or .sum(). Downsampling with methods like [`coarsen()`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.coarsen.html) reduces data resolution to improve performance, match coarser grids, or reduce noise. 
For example, to lower a resolution from 0.25° to 1°, use a window size of 4 and apply .mean(). Remember to update metadata, such as units, afterwards.     


**5. Exercise**:  
1. Average the data onto a 1°x1° grid. Name it `coarsened_mean`
2. Update attributes
3. Plot the results

In [ ]:
# your calculation here
coarsened_mean = 

In [ ]:
# your plot here



In the following, we plot the wind vectors together with the absolute wind speed.

In [ ]:
# run the cell again with quiver
#coarsened_mask = mask.coarsen(lat=4, lon=4).mean()
# Create the figure and axis with Robinson projection
fig, ax = plt.subplots(
    figsize=(12, 7),
    subplot_kw={'projection': ccrs.Robinson()}  # map projection
)

# 1) Plot the coarsened mean
#    Tip: Adjust cmap or levels for the new variable if needed.

coarsened_mean.where(coarsened_mask).plot(
    ax=ax,
    cmap='viridis',                 
    transform=ccrs.PlateCarree()     # data is lon/lat
)

# 2) Land/border styling — learners only tweak parameters below
ax.add_feature(cfeature.LAND, facecolor='lightgray')     # <- land color
ax.coastlines(linewidth=1.0, color='black')              # <- coastline color/width
ax.add_feature(cfeature.BORDERS, edgecolor='dimgray', linewidth=0.6)  # <- borders

# 3) Gridlines (parallels/meridians): color, linewidth, spacing
gl = ax.gridlines(
    draw_labels=True,        
    linestyle='--',
    color='gray',             
    linewidth=0.6,            
    alpha=0.8
)

gl.top_labels = False
gl.right_labels = False

sample_step_x = 20  # adjust step 
sample_step_y = 40  # adjust step 

U = ds.eastward_wind.isel(time=0).values[::sample_step_x, ::sample_step_y]
V = ds.northward_wind.isel(time=0).values[::sample_step_x, ::sample_step_y]
X = lon2d[::sample_step_x, ::sample_step_y]
Y = lat2d[::sample_step_x, ::sample_step_y]

# Quiver on a map: provide data CRS via 'transform' 
q = ax.quiver(
    X, Y, U, V,
    transform=ccrs.PlateCarree(),  # data are in lon/lat
    scale=500,                     # adjust arrow length scaling
    width=0.0025                   # arrow line width
)



# 4) Title/units: reflect the new variable
time = str(ds.coords['time'].values[0])
ax.set_title(f"Mean Wind Speed {time[:10]} ({ds.northward_wind.units})")

plt.show()

We use the gridded wind vectors and coarsened wind speed for this plot. Ideally, the wind vectors should also be coarsened instead of simply sliced to ensure better comparability with the coarsened wind speed. Despite this, the plot clearly shows regions with higher wind speeds and more prominent wind vectors, while in the blue-shaded areas with lower speeds, the arrows are also smaller.

**Extra Exercise: Coarsen the zonal and meridional wind vectors and plot them together with the coarsened wind speed.**

In [ ]:
# Extra Excercise

Plotting is more efficient on a coarser grid. However, is it always appropriate? Compute the standard deviation within each grid cell and assess the variability that might be obscured due to the coarser gridding.  

**6. Exercise:**  
1. Compute standard deviation (std) of `wind_speed` within each 1°x1° grid cell and name it `coarsened_std`
2. Update attributes
3. Visualize the std using the colormap `cividis`

In [ ]:
# your calculation here 



In [ ]:
# your plot here



**Note:** The plot shows spatial variability in certain areas. The right grid resolution is crucial and varies based on whether you're studying broad or fine details, relevant in both atmospheric and ocean data. 

To summarize, both `slicing` and `coarsen` offer distinct methodologies for handling and visualizing large datasets. While slicing is a direct approach to selectively display data, making visualizations more intelligible, coarsen provides a more comprehensive representation by spatially averaging the data. This ensures key information is retained, but the spatial variability within the data is smoothed out. This is because it averages over specified spatial windows, and as a result, finer-scale variations that fall below the size of this window are effectively lost or averaged out.  

For big patterns, a coarser grid works. For detailed studies on small phenomena, use a fine grid and perhaps zoom into an area of interest.  To achieve this, we can slice the data. Unlike before, where you might pick any x-value for latitude or longitude, we can now select a specific box using `wind_speed_sliced = wind_speed.sel(lon=slice(lon min, lon_max),lat=slice(lat_min, lat_max))`. 


In [ ]:
#run the cell

# Define the Indian Ocean region: 
# Approximate bounds: Longitude 20°E to 120°E, Latitude -60°S to 30°N
lon_min, lon_max = 20, 120
lat_min, lat_max = -60, 30

# Slice the wind speed data for the Indian Ocean region
wind_speed_sliced = wind_speed.sel(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))
mask_sliced = mask.sel(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))

fig, ax = plt.subplots(
    figsize=(12, 5),
    subplot_kw={'projection': ccrs.PlateCarree()}  # geographic coordinates (lon/lat)
)

# Plot eastward wind field from the xarray dataset
wind_speed_sliced.where(mask_sliced).plot(
    ax=ax,
    cmap='coolwarm',
    transform=ccrs.PlateCarree()  # data is already in lon/lat
)

# Add coastlines and borders
ax.coastlines(linewidth=1, color='grey')        # color of coastlines
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')  # color of borders
ax.add_feature(cfeature.LAND, facecolor='white')  # color of landmasses

# Add gridlines with labels
gl = ax.gridlines(
    draw_labels=True,
    linestyle='--',
    color='gray',
    alpha=0.7
)
gl.top_labels = False
gl.right_labels = False

# Set a dynamic title based on the dataset
plt.title(f"Wind Speed over Indian Ocean at {time[:10]} ({wind_speed_sliced.units})")

# Show the plot
plt.show()


**Extra Exercise: Slice a region in the Northern North Atlantic.**

In [ ]:
#Extra Exercise



In [ ]:
# save data as netcdf
#wind_speed.to_netcdf('wind_speed_012024.nc')

In [ ]:
ds.close()

# Key Learnings:


**Scientific Modules:** You've worked with numpy and xarray, both crucial in handling and analyzing scientific data in Python.

**Variables and Types:** You've interacted with various data types like floats, arrays, and DataArrays in xarray, understanding how to manipulate and process them.

**Operators and Comparisons:** You've used mathematical operations in slicing and transforming data and made comparisons when selecting regions (e.g., slicing wind speed data by latitude and longitude).

**Linear Algebra:** You've dealt with vector data, like the zonal and meridional wind components, and their role in wind vector calculations.

**Scientific Algorithms:** You've calculated means and standard deviations (e.g., resampling or coarsening data) to summarize wind speed data, a fundamental part of statistical analysis.

**Exceptions and Error Handling:**You navigated errors like ValueError and projection issues when plotting, which required adjusting your approach to ensure the code executed properly.

**Visualization, Plotting, and Data Organization:** You’ve extensively worked on visualization tasks using matplotlib and Basemap to represent data with wind vector plots and contour maps. You've also made adjustments to plot wind speed and zonal/meridional wind components.

**Data Extraction and Manipulation:**You've sliced and extracted specific regions of datasets, e.g., focusing on the Indian Ocean for wind speed visualization. You’ve also resampled data spatially using xarray operations.

**Spatial Data Resampling:** You’ve applied resampling techniques (coarsening) to reduce the resolution of data for more manageable computations and better visualization.
